# Reference
1. [NestedTensor](https://pytorch.org/tutorials/prototype/nestedtensor.html)

In [1]:
import numpy as np
import timeit
import torch
import torch.nn.functional as F

from torch import nn

torch.manual_seed(1)
np.random.seed(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
N = 3
E_q, E_k, E_v, E_total = 512, 512, 512, 512
E_out = E_q
nheads = 8

In [3]:
dropout_p = 0.0

In [4]:
class MultiHeadAttention(nn.Module):
  """
  Computes multi-head attention. Supports nested or padded tensors.

  Args:
      E_q (int): Size of embedding dim for query
      E_k (int): Size of embedding dim for key
      E_v (int): Size of embedding dim for value
      E_total (int): Total embedding dim of combined heads post input projection. Each head
          has dim E_total // nheads
      nheads (int): Number of heads
      dropout_p (float, optional): Dropout probability. Default: 0.0
  """
  def __init__(self, E_q: int, E_k: int, E_v: int, E_total: int,
                nheads: int, dropout_p: float = 0.0):
    super().__init__()
    self.nheads = nheads
    self.dropout_p = dropout_p
    self.query_proj = nn.Linear(E_q, E_total)
    self.key_proj = nn.Linear(E_k, E_total)
    self.value_proj = nn.Linear(E_v, E_total)
    E_out = E_q
    self.out_proj = nn.Linear(E_total, E_out)
    assert E_total % nheads == 0, "Embedding dim is not divisible by nheads"
    self.E_head = E_total // nheads

  def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
      """
      Forward pass; runs the following process:
          1. Apply input projection
          2. Split heads and prepare for SDPA
          3. Run SDPA
          4. Apply output projection

      Args:
          query (torch.Tensor): query of shape (N, L_t, E_q)
          key (torch.Tensor): key of shape (N, L_s, E_k)
          value (torch.Tensor): value of shape (N, L_s, E_v)

      Returns:
          attn_output (torch.Tensor): output of shape (N, L_t, E_q)
      """
      # Step 1. Apply input projection
      # TODO: demonstrate packed projection
      query = self.query_proj(query)
      key = self.key_proj(key)
      value = self.value_proj(value)

      # Step 2. Split heads and prepare for SDPA
      # reshape query, key, value to separate by head
      # (N, L_t, E_total) -> (N, L_t, nheads, E_head) -> (N, nheads, L_t, E_head)
      query = query.unflatten(-1, [self.nheads, self.E_head]).transpose(1, 2)
      # (N, L_s, E_total) -> (N, L_s, nheads, E_head) -> (N, nheads, L_s, E_head)
      key = key.unflatten(-1, [self.nheads, self.E_head]).transpose(1, 2)
      # (N, L_s, E_total) -> (N, L_s, nheads, E_head) -> (N, nheads, L_s, E_head)
      value = value.unflatten(-1, [self.nheads, self.E_head]).transpose(1, 2)

      # Step 3. Run SDPA
      # (N, nheads, L_t, E_head)
      attn_output = F.scaled_dot_product_attention(
          query, key, value, dropout_p=dropout_p, is_causal=True)
      # (N, nheads, L_t, E_head) -> (N, L_t, nheads, E_head) -> (N, L_t, E_total)
      attn_output = attn_output.transpose(1, 2).flatten(-2)

      # Step 4. Apply output projection
      # (N, L_t, E_total) -> (N, L_t, E_out)
      attn_output = self.out_proj(attn_output)

      return attn_output

In [5]:
def zipf_sentence_lengths(alpha: float, batch_size: int) -> torch.Tensor:
  # generate fake corpus by unigram Zipf distribution
  # from wikitext-2 corpus, we get rank "." = 3, "!" = 386, "?" = 858
  sentence_lengths = np.empty(batch_size, dtype=int)
  for ibatch in range(batch_size):
    sentence_lengths[ibatch] = 1
    word = np.random.zipf(alpha)
    while word != 3 and word != 386 and word != 858:
      sentence_lengths[ibatch] += 1
      word = np.random.zipf(alpha)
  return torch.tensor(sentence_lengths)

zipf_sentence_lengths(2, 2)

tensor([15, 12])

In [6]:
def gen_batch(N, E_q, E_k, E_v, device):
  # generate semi-realistic data using Zipf distribution for sentence lengths
  sentence_lengths = zipf_sentence_lengths(alpha=1.2, batch_size=N)

  # Note: the torch.jagged layout is a nested tensor layout that supports a single ragged
  # dimension and works with torch.compile. The batch items each have shape (B, S*, D)
  # where B = batch size, S* = ragged sequence length, and D = embedding dimension.
  query = torch.nested.nested_tensor([
    torch.randn(l.item(), E_q, device=device)
    for l in sentence_lengths
  ], layout=torch.jagged)

  key = torch.nested.nested_tensor([
    torch.randn(s.item(), E_k, device=device)
    for s in sentence_lengths
  ], layout=torch.jagged)

  value = torch.nested.nested_tensor([
    torch.randn(s.item(), E_v, device=device)
    for s in sentence_lengths
  ], layout=torch.jagged)

  return query, key, value, sentence_lengths

query1, key1, value1, sentence_lengths1 = gen_batch(N, E_q, E_k, E_v, device)
query2, key2, value2, sentence_lengths2 = gen_batch(N, E_q, E_k, E_v, device)
print(f"query1\n{query1}")
print(f"key1\n{key1}")
print(f"value1\n{value1}")
print(f"sequence lengths1\n{sentence_lengths1}")

print(f"query2\n{query2}")
print(f"key2\n{key2}")
print(f"value2\n{value2}")
print(f"sequence lengths2\n{sentence_lengths2}")

query1
NestedTensor(size=(3, j1, 512), offsets=tensor([ 0, 33, 38, 75], device='cuda:0'), contiguous=True)
key1
NestedTensor(size=(3, j2, 512), offsets=tensor([ 0, 33, 38, 75], device='cuda:0'), contiguous=True)
value1
NestedTensor(size=(3, j3, 512), offsets=tensor([ 0, 33, 38, 75], device='cuda:0'), contiguous=True)
sequence lengths1
tensor([33,  5, 37])
query2
NestedTensor(size=(3, j4, 512), offsets=tensor([ 0, 14, 19, 45], device='cuda:0'), contiguous=True)
key2
NestedTensor(size=(3, j5, 512), offsets=tensor([ 0, 14, 19, 45], device='cuda:0'), contiguous=True)
value2
NestedTensor(size=(3, j6, 512), offsets=tensor([ 0, 14, 19, 45], device='cuda:0'), contiguous=True)
sequence lengths2
tensor([14,  5, 26])


In [7]:
def jagged_to_padded(jt, padding_val):
  # TODO: do jagged -> padded directly when this is supported
  return torch.nested.to_padded_tensor(
    torch.nested.nested_tensor(list(jt.unbind())),
    padding_val)

padded_query, padded_key, padded_value = (
  jagged_to_padded(t, 0.0) for t in (query1, key1, value1)
)
print(f"padded_query\n{padded_query}")
print(f"padded_key\n{padded_key}")
print(f"padded_value\n{padded_value}")

padded_query
tensor([[[-0.2963,  2.6764, -0.1408,  ..., -0.3286,  0.1592, -1.2010],
         [ 1.2503,  0.5466, -0.5461,  ...,  0.9187,  0.9484,  1.4460],
         [ 0.1486, -2.4969,  1.6598,  ..., -0.6565,  2.1186,  1.2463],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],

        [[-0.6340, -0.6431, -0.6420,  ..., -0.5100, -0.1409,  1.1841],
         [-0.3568, -2.2259, -1.3494,  ..., -0.1865, -0.2386, -0.7401],
         [-0.1125,  0.2543, -1.0514,  ...,  0.1216, -0.3644,  1.0189],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],

        [[-0.5810, -1.0216,  0.8331,  ..., -0.6628,  0.3443,  1.8825],
         [-0.3967,  1.5503,  0.5

/usr/local/lib/python3.11/dist-packages/torch/nested/__init__.py:220: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [8]:
mha = MultiHeadAttention(E_q, E_k, E_v, E_total, nheads, dropout_p).to(device=device)

In [9]:
def benchmark(func, *args, **kwargs):
  torch.cuda.synchronize()
  begin = timeit.default_timer()
  output = func(*args, **kwargs)
  torch.cuda.synchronize()
  end = timeit.default_timer()
  return output, (end - begin)

output_nested, time_nested = benchmark(mha, query1, key1, value1)
output_padded, time_padded = benchmark(mha, padded_query, padded_key, padded_value)

# padding-specific step: remove output projection bias from padded entries for fair comparison
for i, entry_length in enumerate(sentence_lengths1):
    output_padded[i, entry_length:] = 0.0

print("=== without torch.compile ===")
print("nested and padded calculations differ by", (jagged_to_padded(output_nested, 0.0) - output_padded).abs().max().item())
print("nested tensor multi-head attention takes", time_nested, "seconds")
print("padded tensor multi-head attention takes", time_padded, "seconds")

# warm up compile first...
compiled_mha = torch.compile(mha)
compiled_mha(query1, key1, value1)
compiled_mha(query2, key2, value2)
# ...now benchmark
compiled_output_nested, compiled_time_nested = benchmark(
    compiled_mha, query1, key1, value1)

# warm up compile first...
compiled_mha(padded_query, padded_key, padded_value)
# ...now benchmark
compiled_output_padded, compiled_time_padded = benchmark(
    compiled_mha, padded_query, padded_key, padded_value)

# padding-specific step: remove output projection bias from padded entries for fair comparison
for i, entry_length in enumerate(sentence_lengths1):
    compiled_output_padded[i, entry_length:] = 0.0

print("=== with torch.compile ===")
print("nested and padded calculations differ by", (jagged_to_padded(compiled_output_nested, 0.0) - compiled_output_padded).abs().max().item())
print("nested tensor multi-head attention takes", compiled_time_nested, "seconds")
print("padded tensor multi-head attention takes", compiled_time_padded, "seconds")

=== without torch.compile ===
nested and padded calculations differ by 0.0
nested tensor multi-head attention takes 0.08065653406083584 seconds
padded tensor multi-head attention takes 0.0007458552718162537 seconds


/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:150: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


=== with torch.compile ===
nested and padded calculations differ by 0.0
nested tensor multi-head attention takes 0.0041188690811395645 seconds
padded tensor multi-head attention takes 0.00036082789301872253 seconds
